# 01 - ARDL Model and Bounds Test (Pesaran-Shin-Smith)

This notebook covers the **Autoregressive Distributed Lag (ARDL)** model and the
**bounds test** for cointegration using `chronobox`.

## Topics covered

1. ARDL(p, q₁, q₂, …) model specification
2. Automatic lag selection via AIC/BIC
3. Bounds test (Pesaran, Shin & Smith, 2001)
4. Interpreting I(0)/I(1) critical value bands
5. Long-run and short-run relationships
6. When to use ARDL vs Johansen
7. Exercises

---

### The ARDL Model

The ARDL(p, q₁, …, qₖ) model includes lags of both the dependent variable and
exogenous regressors:

$$y_t = c + \sum_{i=1}^{p} \phi_i \, y_{t-i} + \sum_{j=1}^{k} \sum_{l=0}^{q_j} \beta_{jl} \, x_{j,t-l} + u_t$$

where:
- $p$ is the number of AR lags of $y$
- $q_j$ is the number of distributed lags of each regressor $x_j$
- $\phi_i$ are the AR coefficients
- $\beta_{jl}$ are the distributed lag coefficients

### Why ARDL?

The ARDL approach has key advantages over the Johansen method:

| Feature | ARDL/Bounds Test | Johansen |
|---------|-----------------|----------|
| Integration order | Works with I(0), I(1), or mix | Requires all I(1) |
| Sample size | Better in small samples | Needs large T |
| Variables | Single equation (y is dependent) | System estimation |
| Pre-testing | No need for unit root pre-tests | Must confirm I(1) first |

**Reference:** Pesaran, M. H., Shin, Y., & Smith, R. J. (2001). *Bounds testing approaches to the analysis of level relationships.* Journal of Applied Econometrics, 16(3), 289–326.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

from chronobox.models.ardl import ARDL

sys.path.insert(0, os.path.join("..", "utils"))
from plot_helpers import (
    plot_ardl_lag_structure,
    plot_bounds_test,
    plot_long_run_relationship,
)

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
np.set_printoptions(precision=4, suppress=True)

print("All imports loaded successfully.")

## 1. Loading the data

We use the `ardl_synthetic.csv` dataset, which contains 4 variables with **mixed
orders of integration** — the ideal scenario for the ARDL bounds test:

- `y`: I(1) — dependent variable, cointegrated with x1
- `x1`: I(1) — cointegrated with y (true long-run: y = 1.5 + 0.6·x1)
- `x2`: I(0) — stationary regressor (short-run effects only)
- `x3`: I(1) — independent random walk (not cointegrated with y)

In [ ]:
data_path = os.path.join("..", "data", "ardl_synthetic.csv")
df = pd.read_csv(data_path, parse_dates=["date"])

print(f"Shape: {df.shape}")
print(f"Period: {df['date'].iloc[0].date()} to {df['date'].iloc[-1].date()}")
print(f"\nDescriptive statistics:")
print(df[["y", "x1", "x2", "x3"]].describe().round(4))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
fig.suptitle("ARDL Synthetic Dataset — Mixed Integration Orders", fontsize=14)

for ax, col, label in zip(
    axes.flat,
    ["y", "x1", "x2", "x3"],
    ["y — I(1), dependent", "x1 — I(1), cointegrated with y",
     "x2 — I(0), stationary", "x3 — I(1), not cointegrated"],
):
    ax.plot(df["date"], df[col], linewidth=1.0)
    ax.set_title(label, fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "ardl_data_overview.png"), bbox_inches="tight")
plt.show()

print("Note: x2 fluctuates around zero (stationary), while y, x1, x3 show trends (I(1)).")

## 2. ARDL Model Specification and Lag Selection

The ARDL model requires choosing the number of lags $p$ for $y$ and $q_j$ for each
regressor $x_j$. `chronobox` supports automatic selection via information criteria:

- **AIC** (Akaike): tends to select more lags, better for forecasting
- **BIC** (Schwarz): penalizes complexity more, better for identification

The search grid tests all combinations of $p \in [1, \text{max\_p}]$ and
$q \in [0, \text{max\_q}]$, selecting the specification that minimizes the criterion.

In [ ]:
# Prepare data arrays
y = df["y"].values
x = df[["x1", "x2", "x3"]].values

# Automatic lag selection via AIC
ardl_aic = ARDL(max_p=4, max_q=4, criterion="aic")
result_aic = ardl_aic.fit(y, x)

print("=" * 50)
print("AIC-selected model:")
print(f"  ARDL({result_aic.y_lags}, {', '.join(map(str, result_aic.x_lags))})")
print(f"  AIC = {result_aic.aic:.4f}")
print(f"  BIC = {result_aic.bic:.4f}")
print(f"  R² = {result_aic.r_squared:.4f}")

# Automatic lag selection via BIC
ardl_bic = ARDL(max_p=4, max_q=4, criterion="bic")
result_bic = ardl_bic.fit(y, x)

print(f"\nBIC-selected model:")
print(f"  ARDL({result_bic.y_lags}, {', '.join(map(str, result_bic.x_lags))})")
print(f"  AIC = {result_bic.aic:.4f}")
print(f"  BIC = {result_bic.bic:.4f}")
print(f"  R² = {result_bic.r_squared:.4f}")
print("=" * 50)

In [ ]:
# Full summary of the AIC-selected model
print(result_aic.summary())

In [ ]:
# Compare AIC and BIC across several specifications
specs = []
ardl_search = ARDL(max_p=4, max_q=4)
for p_try in range(1, 5):
    for q_try in range(0, 5):
        try:
            r = ardl_search.fit(y, x, p=p_try, x_lags=[q_try] * 3)
            specs.append({
                "p": p_try, "q": q_try,
                "AIC": r.aic, "BIC": r.bic,
                "R²": r.r_squared, "nparams": r.k_params,
            })
        except ValueError:
            continue

specs_df = pd.DataFrame(specs)
print("Model comparison (uniform q for all regressors):")
print(specs_df.to_string(index=False))

best_aic = specs_df.loc[specs_df["AIC"].idxmin()]
best_bic = specs_df.loc[specs_df["BIC"].idxmin()]
print(f"\nBest by AIC: ARDL({int(best_aic['p'])}, {int(best_aic['q'])}) — AIC = {best_aic['AIC']:.4f}")
print(f"Best by BIC: ARDL({int(best_bic['p'])}, {int(best_bic['q'])}) — BIC = {best_bic['BIC']:.4f}")

## 3. Visualizing the Lag Structure

The ARDL coefficients show how past values of $y$ and $x$ influence the current $y$.
The AR coefficients ($\phi_i$) capture persistence in $y$, while the distributed
lag coefficients ($\beta_{jl}$) capture the dynamic effect of each regressor.

In [ ]:
# Extract AR and x1 coefficients for visualization
p = result_aic.y_lags
coef = result_aic.coefficients

# Layout: [const, y_{t-1}..y_{t-p}, x1_t..x1_{t-q1}, x2_t..x2_{t-q2}, x3_t..x3_{t-q3}]
ar_coefs = coef[1:1 + p]
idx = 1 + p
x1_coefs = coef[idx:idx + result_aic.x_lags[0] + 1]

fig = plot_ardl_lag_structure(
    ar_coefs, x1_coefs,
    var_names=("y", "x1"),
    title=f"ARDL({result_aic.y_lags}, {', '.join(map(str, result_aic.x_lags))}) Lag Structure"
)
plt.savefig(os.path.join("..", "outputs", "ardl_lag_structure.png"), bbox_inches="tight")
plt.show()

## 4. Bounds Test for Cointegration (PSS Test)

The **Pesaran-Shin-Smith (PSS) bounds test** tests for a long-run level relationship.
It is based on the error correction representation of the ARDL:

$$\Delta y_t = c + \pi_{yy} \, y_{t-1} + \boldsymbol{\pi}_{yx}' \, \mathbf{x}_{t-1}
  + \sum_{i=1}^{p-1} \gamma_i \, \Delta y_{t-i}
  + \sum_{j=1}^{k} \sum_{l=0}^{q_j-1} \delta_{jl} \, \Delta x_{j,t-l} + u_t$$

### Hypotheses

- **H₀**: $\pi_{yy} = 0$ and $\boldsymbol{\pi}_{yx} = \mathbf{0}$ (no long-run relationship)
- **H₁**: $\pi_{yy} \neq 0$ or $\boldsymbol{\pi}_{yx} \neq \mathbf{0}$ (long-run relationship exists)

### Critical Value Bands I(0) / I(1)

The key innovation of PSS is that critical values come in **pairs**:

- **Lower bound I(0)**: assumes all regressors are stationary — I(0)
- **Upper bound I(1)**: assumes all regressors are integrated — I(1)

**Decision rules:**
- F > upper bound → **Reject H₀**: cointegration exists (regardless of integration orders)
- F < lower bound → **Cannot reject H₀**: no cointegration
- Lower < F < upper → **Inconclusive**: integration orders matter, need unit root tests

In [ ]:
# Run bounds test on AIC-selected model
bt = result_aic.bounds_test()

print("Pesaran-Shin-Smith Bounds Test")
print("=" * 55)
print(f"F-statistic:  {bt['f_statistic']:.4f}")
print(f"k (regressors): {bt['k']}")
print("-" * 55)
print(f"{'Level':>10}  {'I(0) Lower':>12}  {'I(1) Upper':>12}  {'Decision':>14}")
print("-" * 55)

for level in ["10", "5", "1"]:
    lo = bt[f"lower_{level}"]
    hi = bt[f"upper_{level}"]
    f = bt["f_statistic"]
    if f > hi:
        dec = "Reject H0"
    elif f < lo:
        dec = "Fail to reject"
    else:
        dec = "Inconclusive"
    print(f"{level + '%':>10}  {lo:>12.4f}  {hi:>12.4f}  {dec:>14}")

print("=" * 55)
print(f"\nConclusion at 5%: {bt['conclusion']}")

In [ ]:
# Visualize the bounds test result
fig = plot_bounds_test(
    f_statistic=bt["f_statistic"],
    i0_bound=bt["lower_5"],
    i1_bound=bt["upper_5"],
    significance="5%",
    title="PSS Bounds Test — ARDL with Mixed I(0)/I(1) Regressors",
)
plt.savefig(os.path.join("..", "outputs", "bounds_test_result.png"), bbox_inches="tight")
plt.show()

### Interpreting the I(0)/I(1) Bands

The critical value bands account for uncertainty about the integration order of regressors:

- If **F > I(1) upper bound**, we reject H₀ regardless of whether regressors are
  I(0) or I(1). This is the beauty of the bounds test: **no need for unit root pre-testing**.

- If **F < I(0) lower bound**, we fail to reject even under the most favorable
  scenario (all regressors stationary). No long-run relationship.

- If **I(0) < F < I(1)**, the result depends on the actual integration orders.
  In this case, unit root tests (ADF, KPSS) are needed to resolve ambiguity.

In our synthetic dataset:
- x1 is I(1) and cointegrated with y → the bounds test should detect this
- x2 is I(0) → valid regressor in ARDL regardless
- x3 is I(1) but not cointegrated → may reduce power but test remains valid

This mixed-integration scenario is exactly where **ARDL has an advantage over Johansen**,
which requires all variables to be I(1).

## 5. Long-Run Relationship

When the bounds test rejects H₀, we can estimate the **long-run coefficients**.
The long-run multiplier of $x_j$ is:

$$\theta_j = \frac{\sum_{l=0}^{q_j} \beta_{jl}}{1 - \sum_{i=1}^{p} \phi_i}$$

This tells us the permanent effect of a one-unit change in $x_j$ on $y$ after
all dynamics have played out.

In [ ]:
# Long-run coefficients
lr = result_aic.long_run_coefficients
var_names = ["x1", "x2", "x3"]

print("Long-run coefficients (theta_j):")
print("-" * 40)
for name, coef in zip(var_names, lr):
    print(f"  {name}: {coef:.4f}")

print("\n--- Interpretation ---")
print(f"  True long-run coef of x1: 0.6000")
print(f"  Estimated:                {lr[0]:.4f}")
print(f"  x2 is I(0) — its long-run effect should be modest (short-run only in DGP)")
print(f"  x3 is not cointegrated — its long-run coefficient has no equilibrium meaning")

In [ ]:
# Plot long-run relationship between y and x1
fig = plot_long_run_relationship(
    y=df["y"].values,
    x=df["x1"].values,
    long_run_coef=lr[0],
    var_names=("y", "x1"),
    title="Long-Run Equilibrium: y vs x1",
)
plt.savefig(os.path.join("..", "outputs", "ardl_long_run_relationship.png"), bbox_inches="tight")
plt.show()

print(f"The scatter plot shows the cointegrating relationship between y and x1.")
print(f"The red line represents the estimated long-run equilibrium.")

## 6. ARDL vs Johansen: When to Use Which?

| Criterion | Use ARDL/Bounds Test | Use Johansen |
|-----------|---------------------|-------------|
| **Integration orders** | Mixed I(0)/I(1) or uncertain | All variables confirmed I(1) |
| **Sample size** | Small samples (T < 100) | Large samples (T > 100) |
| **Number of relations** | Single cointegrating relation | Multiple cointegrating relations |
| **Endogeneity** | y is clearly dependent | Symmetric system, no clear dependent var |
| **Pre-testing** | No unit root pre-test needed | Must pre-test for I(1) |

**Key advantage of ARDL:** In practice, unit root tests have low power and can
disagree (ADF says I(1), KPSS says I(0)). The bounds test sidesteps this problem
by providing valid inference regardless of the integration order, as long as no
variable is I(2) or higher.

**Key advantage of Johansen:** When there may be multiple cointegrating relationships
among $K$ variables ($r > 1$), Johansen can detect all of them simultaneously.
ARDL is limited to testing a single long-run relationship with $y$ as dependent.

## Exercicio

### Exercicio 1: Bounds test with US macro data

Use the `us_macro_quarterly.csv` dataset to test for a long-run relationship
between GDP growth and inflation, controlling for fed_funds and unemployment.

Steps:
1. Load the data and set `y = gdp`, `x = [inflation, fed_funds, unemployment]`
2. Fit an ARDL model with AIC selection (`max_p=4, max_q=4`)
3. Run the bounds test and interpret the F-statistic
4. Compute the long-run coefficients
5. Discuss: does a long-run relationship exist between GDP and inflation?

In [ ]:
# --- Exercicio 1: Seu codigo aqui ---

us_df = pd.read_csv(os.path.join("..", "data", "us_macro_quarterly.csv"), parse_dates=["date"])

y_us = us_df["gdp"].values
x_us = us_df[["inflation", "fed_funds", "unemployment"]].values

# Fit ARDL
ardl_us = ARDL(max_p=4, max_q=4, criterion="aic")
result_us = ardl_us.fit(y_us, x_us)

print(f"Selected: ARDL({result_us.y_lags}, {', '.join(map(str, result_us.x_lags))})")
print(result_us.summary())

# Bounds test
bt_us = result_us.bounds_test()
print(f"\nBounds test F-statistic: {bt_us['f_statistic']:.4f}")
print(f"Conclusion at 5%: {bt_us['conclusion']}")

# Long-run coefficients
lr_us = result_us.long_run_coefficients
for name, coef in zip(["inflation", "fed_funds", "unemployment"], lr_us):
    print(f"  Long-run {name}: {coef:.4f}")

### Exercicio 2: AIC vs BIC comparison

Using the synthetic dataset:
1. Fit ARDL with only x1 as regressor (bivariate case)
2. Compare AIC vs BIC lag selection
3. Run bounds test on both — does the conclusion change?
4. With k=1 regressor, the critical values change. Compare with k=3.

In [ ]:
# --- Exercicio 2: Seu codigo aqui ---

y_biv = df["y"].values
x_biv = df[["x1"]].values

# AIC
r_aic_biv = ARDL(max_p=4, max_q=4, criterion="aic").fit(y_biv, x_biv)
bt_aic_biv = r_aic_biv.bounds_test()
print(f"AIC: ARDL({r_aic_biv.y_lags}, {r_aic_biv.x_lags[0]})")
print(f"  F-stat = {bt_aic_biv['f_statistic']:.4f}, conclusion: {bt_aic_biv['conclusion']}")
print(f"  I(0) 5% = {bt_aic_biv['lower_5']:.3f}, I(1) 5% = {bt_aic_biv['upper_5']:.3f}")

# BIC
r_bic_biv = ARDL(max_p=4, max_q=4, criterion="bic").fit(y_biv, x_biv)
bt_bic_biv = r_bic_biv.bounds_test()
print(f"\nBIC: ARDL({r_bic_biv.y_lags}, {r_bic_biv.x_lags[0]})")
print(f"  F-stat = {bt_bic_biv['f_statistic']:.4f}, conclusion: {bt_bic_biv['conclusion']}")
print(f"  I(0) 5% = {bt_bic_biv['lower_5']:.3f}, I(1) 5% = {bt_bic_biv['upper_5']:.3f}")

print(f"\nNote: with k=1, critical values are higher ({bt_aic_biv['lower_5']:.3f}/{bt_aic_biv['upper_5']:.3f})")
print(f"      vs k=3: ({bt['lower_5']:.3f}/{bt['upper_5']:.3f})")
print(f"More regressors → lower critical values → easier to reject H0.")

---

## Resumo

Neste notebook aprendemos:

1. O modelo **ARDL(p, q₁, …, qₖ)** inclui lags distribuidos de variaveis exogenas
2. A **selecao automatica de lags** via AIC/BIC encontra a melhor especificacao
3. O **bounds test (PSS)** testa cointegracao sem exigir que todas as variaveis sejam I(1)
4. As **bandas I(0)/I(1)** permitem inferencia valida com ordens de integracao mistas
5. Os **coeficientes de longo prazo** $\theta_j$ medem o efeito permanente de $x_j$ sobre $y$
6. O ARDL e preferivel ao Johansen quando ha variaveis com ordens de integracao mistas

No proximo notebook, derivaremos o **Error Correction Model (ECM)** a partir do ARDL
para analisar a dinamica de ajuste ao equilibrio.